# SensorFM — pretraining (resumable, DDP on T4 x2)

**Accelerator: GPU T4 x2**. Settings: **Internet: On**. Secrets (Add-ons → Secrets): `GH_TOKEN`. Input: **Add Input → Your Datasets → `sensorfm-data`**.

**To resume** in a later session: also *Add Input → Your Notebooks → this notebook's previous version*, set `RESUME = True`. The run stops cleanly with a checkpoint before Kaggle's 12 h limit.

In [ ]:
MODEL = 's20m'      # s20m | m60m | l150m
STEPS = 20000       # set from the ladder: scripts/estimate_budget.py
MB = 64             # per-GPU micro-batch
RUN_ID = f'{MODEL}_main'
RESUME = False

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['GH_TOKEN'] = s.get_secret('GH_TOKEN')
try:
    os.environ['HF_TOKEN'] = s.get_secret('HF_TOKEN')
except Exception:
    print('no HF_TOKEN secret (optional)')

In [ ]:
%%bash
set -e
cd /kaggle/working
rm -rf sensorfm-src && git clone -q https://$GH_TOKEN@github.com/DhrumilPanchal10/sensorfm.git sensorfm-src
cd sensorfm-src && git log --oneline -1

In [ ]:
import glob, shutil, os
if RESUME:
    prev = glob.glob(f'/kaggle/input/*/runs/{RUN_ID}')
    assert prev, 'attach the previous version of this notebook as input to resume'
    shutil.copytree(prev[0], f'/kaggle/working/sensorfm-src/runs/{RUN_ID}', dirs_exist_ok=True)
    print('resuming from', prev[0], sorted(os.listdir(prev[0] + '/checkpoints')))

In [ ]:
%%bash -s "$MODEL" "$STEPS" "$MB" "$RUN_ID" "$RESUME"
set -e
cd /kaggle/working/sensorfm-src
MODEL=$1 STEPS=$2 MB=$3 RUN_ID=$4 RESUME=$(echo $5 | tr A-Z a-z) NPROC=2 bash kaggle/pretrain.sh 2>&1 | grep -v Warning | tail -40

In [ ]:
!tail -3 /kaggle/working/runs/*/metrics.jsonl; ls /kaggle/working/runs/*/checkpoints

In [ ]:
# Keep the saved output small: results/ and runs/ were copied to /kaggle/working already.
import shutil; shutil.rmtree('/kaggle/working/sensorfm-src', ignore_errors=True)
!du -sh /kaggle/working/* 2>/dev/null